In [102]:
from click import ClickException
from google import genai
from dotenv import load_dotenv
import os

from google.genai.errors import ClientError

from utils import PDF_DIR

load_dotenv(override=True)

True

In [103]:
print(os.getcwd())

/Users/abrahamlincoln/Documents/ResumeRanker


In [104]:
client=genai.Client(vertexai=True,api_key=os.getenv("gemapikey"))
model="gemini-2.5-flash-lite"

In [105]:
prompt="""You are a resume information extractor.

Extract structured information from the provided resume and return ONLY valid JSON following the exact schema below.

IMPORTANT RULES:

1. Output MUST be valid JSON only. No explanations.
2. Use short keywords instead of sentences wherever possible.
3. Skills must be concise technical keywords.
4. Remove duplicate skills.
5. Normalize experience duration to total months.
6. If a field is missing, return null.
7. Do NOT include long descriptions or paragraphs.
8. Prefer technical terms, tools, frameworks, languages, and measurable impacts.

JSON SCHEMA:

{
"name": "",
"email": "",
"phone": "",
"skills": [],
"experience": [
{
"company": "",
"role_keywords": [],
"skills": [],
"duration_months": 0,
"impact_keywords": []
}
],
"projects": [
{
"skills": [],
"domain_keywords": [],
"skill_intensity_score": 0
}
]
}

FIELD DEFINITIONS:

skills:
List of distinct technical keywords such as programming languages, frameworks, databases, cloud platforms, tools.

role_keywords:
Short role descriptors such as backend, frontend, fullstack, ML, data-engineering, mobile, devops, cloud.

impact_keywords:
Short measurable or technical outcomes such as:
["50k_users", "85_percent_efficiency", "low_latency", "automation", "scaling"]

projects.skills:
Technical skills used in the project.

projects.domain_keywords:
Short domain indicators such as:
["mobile_app", "ai", "cloud", "iot", "web_platform"]

projects.skill_intensity_score:
Integer from 0–100 estimating how heavily the listed skills were used in the project.

Return ONLY the JSON object.
"""

In [106]:
response_dict={}
response_text_dict={}

The code crashed when request failed due to exhausted resources. I added an exponentially increasing delay timer and handled the ClientError exception to resolve this issue.

In [111]:
from google.genai import types
from google.genai.errors import ClientError
import time
for i in range(2,20):
    delay=5
    while True:
        try:
            start=time.perf_counter()
            with open(PDF_DIR/f"{i}.pdf","rb") as temp:
                temp_bytes=temp.read()
            temp_response=client.models.generate_content(model=model,contents=[prompt,types.Part.from_bytes(data=temp_bytes,mime_type="application/pdf")])
            response_dict[f"{i}.pdf"]=temp_response
            response_text_dict[f"{i}.pdf"]=temp_response.text
            end=time.perf_counter()
            print(f"{end-start} s for {i}.pdf")
            break
        except ClientError as error:
            if "RESOURCE_EXHAUSTED" in str(error):
                print(f"Rate ceiling hit. Delaying by {delay} s for {i}.pdf...")
                time.sleep(delay)
                delay*=2
            else:
                raise

21.576353207987268 s for 2.pdf
Rate ceiling hit. Delaying by 5 s for 3.pdf...
Rate ceiling hit. Delaying by 10 s for 3.pdf...
Rate ceiling hit. Delaying by 20 s for 3.pdf...
22.185809208021965 s for 3.pdf
13.996081541001331 s for 4.pdf
25.833471291989554 s for 5.pdf
9.036533292004606 s for 6.pdf
Rate ceiling hit. Delaying by 5 s for 7.pdf...
Rate ceiling hit. Delaying by 10 s for 7.pdf...
17.336672915989766 s for 7.pdf
5.116663541994058 s for 8.pdf
6.0407928340137005 s for 9.pdf
Rate ceiling hit. Delaying by 5 s for 10.pdf...
18.248130207997747 s for 10.pdf
7.676677333016414 s for 11.pdf
4.304825999977766 s for 12.pdf
10.464137458999176 s for 13.pdf
6.732684875023551 s for 14.pdf
5.733196416986175 s for 15.pdf
4.7692503329890314 s for 16.pdf
12.225844000000507 s for 17.pdf
10.548060625005746 s for 18.pdf
9.522921875002794 s for 19.pdf


In [112]:
response_text_dict

{'2.pdf': '```json\n{\n  "name": "Deep M. Mehta",\n  "email": "mail@deepmehta.co.in",\n  "phone": null,\n  "skills": [\n    "C#",\n    "Java",\n    "Python",\n    "Ruby",\n    "Javascript",\n    "SQL",\n    "Angular",\n    "React",\n    "Android",\n    "React-Native",\n    "Django",\n    "Azure",\n    "AWS",\n    "Firebase",\n    "App-Service",\n    "Functions",\n    "AD",\n    "KeyVault",\n    "APIM",\n    "CosmosDB",\n    "Queue",\n    "Redis",\n    "IoT-Hub",\n    "App-Gateway",\n    "Load-Balancer",\n    "VNet",\n    "VPN",\n    "Storage",\n    "Cognitive-Services",\n    "FHIR",\n    "Azure-DevOps",\n    "GitHub Actions",\n    "Terraform",\n    "ARM-Templates",\n    "Ansible",\n    "Kubernetes",\n    "LLMs",\n    "PowerShell",\n    "Unix Shell"\n  ],\n  "experience": [\n    {\n      "company": "Microsoft",\n      "role_keywords": [\n        "Software Engineer"\n      ],\n      "skills": [\n        "Azure-DevOps",\n        "GitHub Actions",\n        "Terraform",\n        "ARM-Templa